In [1]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from src import rules

## Train set მონაცემების ჩატვირთვა

In [2]:
train = pd.read_csv("../data/train.csv")
print(train.shape)
print(rules.explain(train.text.iloc[0]))

(4136, 2)
HAM (score 1.5 / threshold 2): cash_voucher, long_message


### შეფასება. Precision: როცა ფილტრი ამბობს რომ სპამია, რამდენად ხშირად არის მართალი. Recall: რეალური სპამის რა ნაწილია დაჭერილი

In [3]:
preds = rules.predict(train.text)
print(classification_report(train.y, preds, digits=3))
print(confusion_matrix(train.y, preds))


              precision    recall  f1-score   support

           0      0.983     0.998     0.991      3614
           1      0.987     0.879     0.930       522

    accuracy                          0.983      4136
   macro avg      0.985     0.939     0.960      4136
weighted avg      0.983     0.983     0.983      4136

[[3608    6]
 [  63  459]]


### წესები pricing small print, text keyword, opt_out, short_code, mobile_offer არასოდეს ცდებიან. long_message, exclamations და shouting კი ჰემზეც მოქმედებენ, ამიტომ მათი წონა 0.5-მდე არის დაყვანილი

In [4]:
rules.rule_stats(train.text, train.y)

,rule,weight,fires_spam,fires_ham,precision,spam_coverage
1,pricing_small_print,2.0,131,0,1.000000,0.250958
2,text_keyword,2.0,134,0,1.000000,0.256705
3,opt_out,2.0,56,0,1.000000,0.107280
7,short_code,1.0,171,0,1.000000,0.327586
12,mobile_offer,1.0,43,0,1.000000,0.082375
0,phone_number,2.0,242,1,0.995885,0.463602
4,money,1.5,174,3,0.983051,0.333333
8,prize_words,1.0,141,6,0.959184,0.270115
9,claim_words,1.0,82,4,0.953488,0.157088
6,url,1.0,103,8,0.927928,0.197318


### ცრუ დადებითები, რეალური მესიჯები რომლებიც ფილტრმა სპამად მონიშნა.

In [5]:
fp = train[(train.y == 0) & (preds == 1)]
print(len(fp), "false positives")
for t in fp.text.head(10):
    print(rules.explain(t), "|", t)
    print()

6 false positives
SPAM (score 2.5 / threshold 2): url, free, long_message | Hi, Mobile no.  &lt;#&gt;  has added you in their contact list on www.fullonsms.com It s a great place to send free sms to people For more visit fullonsms.com

SPAM (score 2.5 / threshold 2): money, shouting, long_message | "YEH I AM DEF UP4 SOMETHING SAT,JUST GOT PAYED2DAY & I HAVBEEN GIVEN A£50 PAY RISE 4MY WORK & HAVEBEEN MADE PRESCHOOLCO-ORDINATOR 2I AM FEELINGOOD LUV"

SPAM (score 2 / threshold 2): url, free | It doesnt make sense to take it there unless its free. If you need to know more, wikipedia.com

SPAM (score 2 / threshold 2): url, shouting, long_message | Let Ur Heart Be Ur Compass Ur Mind Ur Map Ur Soul Ur Guide And U Will Never loose in world....gnun - Sent via WAY2SMS.COM

SPAM (score 2 / threshold 2): money, long_message | Hi Jon, Pete here, Ive bin 2 Spain recently & hav sum dinero left, Bill said u or ur rents mayb interested in it, I hav 12,000pes, so around £48, tb, James.

SPAM (score 2.5

### ცრუ დადებითების რაოდენობა იყო 16, შემდეგ ჩამოვიყვანე 9 მდე, ბოლო ვერსიით კი 6 მდე. მიზეზები იყოსიტყვა 'won' ემთხვეოდა wont-ს, cash ჩვეულებრივ საუბარშიც გვხვდებოდა, url-ები ზოგჯერ ნამდვილი იყო, მეგობრის ნომერი და ა.შ. სპამს ემთხვეოდა

### წესების თრეინინგი მორჩა და ტესტ სეტი ჯერ არ გამოყენებულა